> **Solutions for `3-probability-theory-lab.ipynb`.** Work through the lab on your own first. Open this notebook to check your answers after you have made a real attempt, or when you have been stuck on a part for a while. Many questions have more than one good answer, so compare your reasoning, not just your wording.

# Lab 3: Probability Theory

> **Course:** Data Science Foundations
> **Lesson:** Day 3 Lab, companion to `3-probability-theory.md` and `3-probability-theory-notebook.ipynb`
> **Dataset:** `titanic` (seaborn). 891 passengers, with survival, sex, age, class and fare. Plus simulated dice.
> **Estimated time:** 90 minutes

## Lab Objectives

- Apply the **complement** and **addition** rules, and tell *mutually exclusive* apart from *independent*
- Compute marginal, joint, and conditional probabilities from real contingency tables
- Test independence from data and explain why $P(A \mid B) \ne P(B \mid A)$
- Use Bayes' theorem and the law of total probability, and verify both against direct counts
- Answer applied questions with `scipy.stats` and confirm each answer by **Monte Carlo simulation**
- Plot a **likelihood function** and find the maximum likelihood estimate

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

titanic = sns.load_dataset("titanic")
print(f"Shape: {titanic.shape}")
titanic[["survived", "sex", "age", "class", "fare", "embark_town"]].head()

## Part 1: The Basic Rules (extension)

Three rules the lesson used without naming:

| Rule | Formula |
|---|---|
| Complement | $P(\text{not } A) = 1 - P(A)$ |
| Addition (either event) | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Mutually exclusive | $A$ and $B$ cannot both happen, so $P(A \cap B) = 0$ |

Mutually exclusive is **not** the same as independent. If two events cannot happen together, knowing one happened tells you a lot about the other.

We roll two fair dice. All 36 outcomes are equally likely:

In [ ]:
outcomes = pd.DataFrame(list(itertools.product(range(1, 7), repeat=2)), columns=["die1", "die2"])
outcomes["total"] = outcomes["die1"] + outcomes["die2"]
print(f"{len(outcomes)} outcomes")
outcomes.head(8)

In [ ]:
p_total_7 = (outcomes["total"] == 7).mean()
p_not_7 = 1 - p_total_7
A = outcomes["die1"] == 6
B = outcomes["total"] >= 10
p_A_and_B = (A & B).mean()
p_A_or_B_direct = (A | B).mean()
p_A_or_B_rule = A.mean() + B.mean() - p_A_and_B

print(f"P(total = 7)        = {p_total_7:.4f}")
print(f"P(total != 7)       = {p_not_7:.4f}")
print(f"P(A), P(B)          = {A.mean():.4f}, {B.mean():.4f}")
print(f"P(A and B)          = {p_A_and_B:.4f}")
print(f"P(A or B) direct    = {p_A_or_B_direct:.4f}")
print(f"P(A or B) by rule   = {p_A_or_B_rule:.4f}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(p_total_7 - 1 / 6) < 1e-12 and abs(p_not_7 - 5 / 6) < 1e-12
assert abs(p_A_or_B_direct - p_A_or_B_rule) < 1e-12 and abs(p_A_or_B_rule - 9 / 36) < 1e-12
print("passed")

In [ ]:
rolls = rng.integers(1, 7, size=(100_000, 2))
total = rolls.sum(axis=1)
sim_7 = np.mean(total == 7)
sim_A_or_B = np.mean((rolls[:, 0] == 6) | (total >= 10))
print(f"P(total = 7): exact {1/6:.4f}  simulated {sim_7:.4f}")
print(f"P(A or B):    exact {9/36:.4f}  simulated {sim_A_or_B:.4f}")

**Question 1.** Are A and B mutually exclusive? Are they independent? Show the numbers.

**Model answer:**

**Not mutually exclusive:** three outcomes, (6,4), (6,5) and (6,6), are in both, so $P(A \cap B) = 3/36 \approx 0.083 \ne 0$.

**Not independent:** $P(A)P(B) = (6/36)(6/36) = 1/36 \approx 0.028$, but the observed $P(A \cap B) = 0.083$ is three times larger. Rolling a 6 first makes a high total much more likely: $P(B \mid A) = 3/6 = 0.5$ vs $P(B) = 0.167$.

## Part 2: Contingency Tables from Real Data

`pd.crosstab` builds the table. Its `normalize` argument turns counts into probabilities:

- `normalize="all"` gives **joint** probabilities (the whole table sums to 1)
- `normalize="index"` gives **conditional on the row** (each row sums to 1)
- `normalize="columns"` gives **conditional on the column** (each column sums to 1)

In [ ]:
counts = pd.crosstab(titanic["sex"], titanic["survived"], margins=True)
display(counts)

joint = pd.crosstab(titanic["sex"], titanic["survived"], normalize="all")
display(joint.round(3))

p_female = (titanic["sex"] == "female").mean()
p_survived = titanic["survived"].mean()

p_surv_given_female = titanic.loc[titanic["sex"] == "female", "survived"].mean()
p_surv_given_male = titanic.loc[titanic["sex"] == "male", "survived"].mean()
p_female_given_surv = (titanic.loc[titanic["survived"] == 1, "sex"] == "female").mean()

print(f"P(female)              = {p_female:.3f}")
print(f"P(survived)            = {p_survived:.3f}")
print(f"P(survived | female)   = {p_surv_given_female:.3f}")
print(f"P(survived | male)     = {p_surv_given_male:.3f}")
print(f"P(female | survived)   = {p_female_given_surv:.3f}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(joint.values.sum() - 1) < 1e-9
assert abs(p_surv_given_female - joint.loc["female", 1] / p_female) < 1e-9
print("passed")

**Question 2.** Put $P(\text{survived} \mid \text{female})$ and $P(\text{female} \mid \text{survived})$ into plain English. Why are they different numbers?

**Model answer:**

$P(\text{survived} \mid \text{female}) \approx 0.74$: *of the women on board, about 74% survived.*
$P(\text{female} \mid \text{survived}) \approx 0.68$: *of the survivors, about 68% were women.*

They share the same joint count (233 women who survived) but divide by different groups: 314 women in the first case, 342 survivors in the second. Swapping the condition changes the question.

## Part 3: Independence and Conditioning on More Than One Variable

In [ ]:
surv_by_class = titanic.groupby("class", observed=True)["survived"].mean()
fig, ax = plt.subplots(figsize=(7, 4))
surv_by_class.plot(kind="bar", ax=ax, color="steelblue")
ax.axhline(p_survived, color="red", ls="--", label=f"overall P(survived) = {p_survived:.2f}")
ax.set_ylabel("P(survived | class)")
ax.set_ylim(0, 1)
ax.legend()
plt.show()

first = titanic["class"] == "First"
observed = (first & (titanic["survived"] == 1)).mean()
expected = first.mean() * p_survived
print(f"P(First and survived) observed          = {observed:.4f}")
print(f"P(First) x P(survived), if independent  = {expected:.4f}")

In [ ]:
rates = titanic.pivot_table(index="sex", columns="class", values="survived", aggfunc="mean", observed=True)
fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(rates, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
ax.set_title("P(survived | sex, class)")
plt.show()

**Question 3.** Is survival independent of class? What does the heatmap add that the sex-only and class-only tables could not show?

**Model answer:**

No. First-class passengers survived at about 63% vs 24% in third class, and the observed joint probability (0.153) is well above the independence prediction (0.093).

The heatmap shows that the two factors combine: women in first and second class survived at over 90%, but women in third class survived at only 50%. Men in second and third class survived at around 14 to 16%. Conditioning on one variable at a time hides these interactions.

## Part 4: Bayes' Theorem on Real Data

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} \qquad\qquad P(B) = \sum_i P(B \mid A_i)\,P(A_i)$$

In [ ]:
p_female_given_surv_bayes = p_surv_given_female * p_female / p_survived
print(f"P(female | survived): Bayes {p_female_given_surv_bayes:.4f}   direct {p_female_given_surv:.4f}")

p_class = titanic["class"].value_counts(normalize=True)
p_survived_total = (surv_by_class * p_class).sum()
print(f"P(survived): total probability {p_survived_total:.4f}   direct {p_survived:.4f}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(p_female_given_surv_bayes - p_female_given_surv) < 1e-9
assert abs(p_survived_total - p_survived) < 1e-9
print("passed")

### 4.3 Updating twice

Recall the lesson's medical test: 1% prevalence, 95% sensitivity, 5% false positive rate, so one positive result gives only a 16% chance of disease. What if the patient takes the test again (independently) and is positive again? **The posterior after the first test becomes the prior for the second.**

In [ ]:
def posterior(prior, sensitivity, false_pos_rate):
    evidence = sensitivity * prior + false_pos_rate * (1 - prior)
    return sensitivity * prior / evidence


belief = 0.01
rows = [{"n_positive_tests": 0, "P_disease": belief}]
for i in range(1, 5):
    belief = posterior(belief, 0.95, 0.05)
    rows.append({"n_positive_tests": i, "P_disease": belief})
updates = pd.DataFrame(rows).round(4)
updates

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(posterior(0.01, 0.95, 0.05) - 0.161) < 0.001
print("passed")

**Question 4.** How many consecutive positive tests does it take before the patient is more likely sick than healthy? Why do doctors order a confirmatory test instead of acting on the first one?

**Model answer:**

Two. The probability goes from 1% to about 16% after one positive and about 78% after two, so it passes 50% on the second test. By three positives it is about 98.6%. The first positive mostly moves a rare prior up to a plausible level. A second, independent positive is much more informative because the false-positive explanation has to happen twice. That is exactly the logic of confirmatory testing. The catch: it only works if the tests' errors are independent, for example a different test method.

## Part 5: Applied Problems, Exact vs Simulated

Solve each problem with `scipy.stats`, then check it by simulating 200,000 draws with `rng`. Fill the `results` table.

1. **Delivery times** are Normal with mean 35 and sd 6 minutes. (a) P(a delivery takes more than 45 min)? (b) What delivery time should the company promise so that 95% of orders arrive on time?
2. **A clinic** receives patients at an average of 3.5 per hour (Poisson). (c) P(at most 2 patients in an hour)? (d) P(more than 10 patients in a *2-hour* window)? *Hint: the rate scales with the interval.*
3. **A quiz** has 20 multiple-choice questions with 4 options each. (e) P(a student who guesses every answer gets at least 10 right)?
4. **A shuttle** leaves at a uniformly random moment in a 15-minute window. (f) P(you wait between 5 and 8 minutes)?

In [ ]:
N = 200_000
delivery = stats.norm(35, 6)
clinic_1h, clinic_2h = stats.poisson(3.5), stats.poisson(7.0)
quiz = stats.binom(20, 0.25)
shuttle = stats.uniform(0, 15)

sim_delivery = rng.normal(35, 6, N)
sim_1h = rng.poisson(3.5, N)
sim_2h = rng.poisson(7.0, N)
sim_quiz = rng.binomial(20, 0.25, N)
sim_wait = rng.uniform(0, 15, N)

results = {
    "a: P(delivery > 45)": (delivery.sf(45), np.mean(sim_delivery > 45)),
    "b: 95th percentile delivery time": (delivery.ppf(0.95), np.percentile(sim_delivery, 95)),
    "c: P(patients <= 2 in 1h)": (clinic_1h.cdf(2), np.mean(sim_1h <= 2)),
    "d: P(patients > 10 in 2h)": (clinic_2h.sf(10), np.mean(sim_2h > 10)),
    "e: P(quiz score >= 10 by guessing)": (quiz.sf(9), np.mean(sim_quiz >= 10)),
    "f: P(5 <= wait <= 8)": (shuttle.cdf(8) - shuttle.cdf(5), np.mean((sim_wait >= 5) & (sim_wait <= 8))),
}
pd.DataFrame(results, index=["exact", "simulated"]).T.round(4)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for label, (exact, sim) in results.items():
    tol = 0.2 if label.startswith("b") else 0.005
    assert abs(exact - sim) < tol, label
print("passed")

**Question 5.** For (d) and (e), why does `sf(10)` answer "more than 10" but `sf(9)` answers "at least 10"? Which kind of distribution makes this detail matter?

**Model answer:**

`sf(k)` is $P(X > k) = 1 - \text{cdf}(k)$, a strict inequality. For a **discrete** distribution, "at least 10" means $X \ge 10$, which is $X > 9$, so you need `sf(9)`. For continuous distributions (Normal, Uniform) a single point has probability 0, so $>$ and $\ge$ give the same answer. Off-by-one errors here are one of the most common bugs in applied probability code.

## Part 6: Likelihood and Maximum Likelihood (extension)

The lesson's example: *you flip a coin 10 times and get 9 heads. How plausible is a fair coin?* The **likelihood** of a candidate heads-probability $p$ is the probability of the observed data if $p$ were true:

$$L(p) = P(\text{9 heads in 10} \mid p) = \binom{10}{9} p^9 (1-p)$$

The data is fixed; $p$ varies.

In [ ]:
p_grid = np.linspace(0, 1, 1001)
likelihood = stats.binom.pmf(9, 10, p_grid)
p_mle = p_grid[np.argmax(likelihood)]
likelihood_ratio = stats.binom.pmf(9, 10, 0.9) / stats.binom.pmf(9, 10, 0.5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_grid, likelihood, lw=2)
ax.axvline(p_mle, color="red", ls="--", label=f"MLE p = {p_mle:.2f}")
ax.axvline(0.5, color="grey", ls=":", label="fair coin p = 0.5")
ax.set_xlabel("p (probability of heads)")
ax.set_ylabel("L(p)")
ax.set_title("Likelihood of p after observing 9 heads in 10 flips")
ax.legend()
plt.show()

print(f"MLE: {p_mle:.3f}")
print(f"L(0.9) / L(0.5) = {likelihood_ratio:.1f}")
print(f"Area under L(p): {np.trapezoid(likelihood, p_grid):.4f}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(p_mle - 0.9) < 1e-9
print("passed")

**Question 6.** The area under the likelihood curve is about 0.09, not 1. Why is that fine? How is this connected to "training" a model?

**Model answer:**

A likelihood is a function of the *parameter* with the data held fixed. It is not a probability distribution over $p$, so nothing requires it to integrate to 1 (probabilities sum to 1 over *outcomes*, not over parameter values). What matters is comparing heights: the data is about 39 times more likely under $p = 0.9$ than under a fair coin.

Picking the parameter with the highest likelihood (here $\hat{p} = 9/10$) is **maximum likelihood estimation**. Training logistic regression, and most other ML models, does the same thing with many parameters. On Saturday you will see the tool that does the maximising: gradient descent.

## Part 7: Wrap-Up

A journalist writes: *"68% of Titanic survivors were women, so a woman had a 68% chance of surviving."* In 3 to 4 sentences, explain the mistake and give the correct figure.

**Model answer:**

The journalist confused $P(\text{female} \mid \text{survived})$ with $P(\text{survived} \mid \text{female})$. 68% describes the make-up of the survivors, not the chances of a woman on board. The correct figure is about 74% (233 of 314 women survived), and it varied hugely by class: over 90% in first and second class, 50% in third. Bayes' theorem converts between the two quantities, and it needs the base rates $P(\text{female})$ and $P(\text{survived})$.

> Submit your completed notebook to the Kanban board under your name by end of day.